In [1]:
import sys
import os

sys.path.append("..")
#sys.path.append(os.path.realpath(os.path.join(os.path.dirname(__file__), '..')))
from quicfire_tools import outputs
from quicfire_tools.parameters import SimulationParameters

import zarr
import pytest
import numpy as np
import pandas as pd
from scipy.io import FortranFile
import xarray as xr
import matplotlib.pyplot as plt
import pickle
import random

from pathlib import Path, PurePath

# DATA_PATH = PurePath("/mnt/c/Users/zacha/Documents/0_Code/quicfire-tools/tests/data")
# SIMULATION_PATH = DATA_PATH.joinpath("test_run_eng")
# OUTPUT_PATH = SIMULATION_PATH.joinpath("Output")
# DRAWFIRE_PATH = SIMULATION_PATH.joinpath("drawfire")
# ZARR_PATH = OUTPUT_PATH.joinpath("outputs.zarr")

# # Create simulation parameters object
# SIM_PARAMS = SimulationParameters(
#     nx=400,
#     ny=200,
#     nz=1,
#     dx=2,
#     dy=2,
#     dz=1,
#     wind_speed=6,
#     wind_direction=270,
#     sim_time=600,
#     auto_kill=0,
#     num_cpus=8,
#     fuel_flag=4,
#     ignition_flag=1,
#     output_time=100,
#     topo_flag=0,
# )

DATA_PATH = PurePath("/mnt/c/Users/zacha/Documents/0_Projects")
SIMULATION_PATH = DATA_PATH.joinpath("0016_FtStewart", "F6_4", "1_Runs", "01_FastFuelsAerialIg531")
OUTPUT_PATH = SIMULATION_PATH.joinpath("Output")
DRAWFIRE_PATH = SIMULATION_PATH.joinpath("drawfire")
ZARR_PATH = OUTPUT_PATH.joinpath("outputs.zarr")

# Create simulation parameters object
SIM_PARAMS = SimulationParameters(
    nx=968,
    ny=1978,
    nz=40,
    dx=2,
    dy=2,
    dz=1,
    wind_speed=6.5,
    wind_direction=270,
    sim_time=4067,
    auto_kill=0,
    num_cpus=8,
    fuel_flag=5,
    ignition_flag=7,
    output_time=100,
    topo_flag=0,
)

def main():
    #Setup drawfire folder:
    if not os.path.exists(DRAWFIRE_PATH):
        os.makedirs(DRAWFIRE_PATH)
    save_dir = DRAWFIRE_PATH
    SMOLDER_THRESHOLD = 25
    #Use library to load and calculate surfEnergy outputs
    simulation_outputs = outputs.SimulationOutputs(OUTPUT_PATH, SIM_PARAMS)

    # output = simulation_outputs.get_output("surfEnergy")
    # arr = simulation_outputs.to_numpy(output)

    # # Create a DataArray object from the numpy array
    # da = xr.DataArray(arr, dims=["time", "y", "x"])

    # # Create a Dataset object from the DataArray object
    # ds = da.to_dataset(name="data")
    if not os.path.exists(ZARR_PATH):
        zarr_file = simulation_outputs.to_zarr(ZARR_PATH)
    zarr.convenience.consolidate_metadata(ZARR_PATH)
    ds = xr.open_zarr(ZARR_PATH)

    #Calc time for max power
    ds = ds.fillna(0) #Convert nan to 0 for dask
    xarr_max_power_time = ds.surfEnergy.argmax('time')
    xarr_max_power = ds.surfEnergy[xarr_max_power_time.compute()]   

    #Calc Total Energy
    xarr_total_energy = ds.surfEnergy.sum(dim='time')

    ###Calc Times: arrival, stop, residence
    ##Removed forloop to improve speed
    #https://stackoverflow.com/questions/47269390/how-to-find-first-non-zero-value-in-every-column-of-a-numpy-array
    #https://stackoverflow.com/questions/66305130/index-of-last-occurence-of-true-in-every-row
    burned_binary = (ds>SMOLDER_THRESHOLD)
    #Arrival time
    xarr_arrival_time = burned_binary.surfEnergy.argmax('time')
    xarr_arrival_time = xr.where(xarr_arrival_time==0,np.nan,xarr_arrival_time) #0 to nan
    xarr_arrival_time = xarr_arrival_time.compute()

    #Fire stop time
    xarr_fire_stop_time = burned_binary.dims['time'] - burned_binary.surfEnergy[::-1,:,:].argmax('time') - 1
    xarr_fire_stop_time = xr.where((burned_binary.surfEnergy[-1,:,:]==0) & (xarr_fire_stop_time==xarr_fire_stop_time.max()),np.nan,xarr_fire_stop_time) #non-burning cells to nan
    xarr_fire_stop_time = xarr_fire_stop_time.compute()
    del burned_binary

    xarr_residence_time = xarr_fire_stop_time - xarr_arrival_time

    #NEW code to run for avg power and stdDev:
    burn_indexes = np.where(xarr_residence_time>0)
    num_burn_cells = len(burn_indexes[0])
    max_res_time = xarr_residence_time.max()
    power_burn_cells = np.empty((num_burn_cells,max_res_time))
    for i in range(num_burn_cells):
        ty = burn_indexes[0][i] #temp y
        tx = burn_indexes[1][i]
        start_t = int(xarr_arrival_time[y_cell, x_cell])
        stop_t = int(xarr_fire_stop_time[y_cell, x_cell])
        cell_power = ds.surfEnergy[start_t:stop_t,y_cell,x_cell]
        power_burn_cells[i,:(stop_t-start_t)] = cell_power.to_numpy()
    
    power_mean_overtime = np.nanmean(power_burn_cells, axis=1)
    power_median_overtime = np.nanmedian(power_burn_cells, axis=1)
    power_stdev_overtime = np.nanstd(power_burn_cells, axis=1)

    #Raw data for Joe:
    df = pd.DataFrame({"power_mean_overtime":power_mean_overtime,"power_median_overtime":power_median_overtime,
                       "power_stdev_overtime":power_stdev_overtime})
    df.to_csv(os.path.join(save_dir,"FtStewart_PowerOvertime.csv"), index=False)
    del df

    plt.plot(max_res_time, power_mean_overtime)
    plt.fill_between(power_mean_overtime,power_mean_overtime-power_stdev_overtime,power_mean_overtime+power_stdev_overtime)
    plt.xlabel('Time (s)')
    plt.ylabel('Power (kW/m^2)')
    plt.title('Average Power of Surface Cells After Ignition')
    plt.savefig(os.path.join(save_dir, 'SufaceCell_MeanPowerAfterIg.png'))
    plt.close()  

    plt.plot(max_res_time, power_median_overtime)
    plt.fill_between(power_median_overtime,power_median_overtime-power_stdev_overtime,power_median_overtime+power_stdev_overtime)
    plt.xlabel('Time (s)')
    plt.ylabel('Power (kW/m^2)')
    plt.title('Median Power of Surface Cells After Ignition')
    plt.savefig(os.path.join(save_dir, 'SufaceCell_MedianPowerAfterIg.png'))
    plt.close()   
    
    ###This method is dumb. I should be able to use np.where to index the burned cells:
    #Sample burning cells
    def find_cells_that_burned(xarr_residence_time, SIM_PARAMS, n=1, time_len=15):
        """
        xarr_residence_time: residence times
        SIM_PARAMS: class of simulation parameters
        n: # of cells to sample
        time_len: length of time to consider cell burned for sample
        """
        PICKLE_PATH = os.path.join(save_dir, 'cell_that_burned.pkl')
        if not os.path.exists(PICKLE_PATH):
            nx = SIM_PARAMS.nx
            ny = SIM_PARAMS.ny
            burned_cells = []
            print('Starting while loop')
            while len(burned_cells) < n:
                temp_x = int(nx*random.random())
                temp_y = int(ny*random.random())
                temp_tup = (temp_x, temp_y)
                if xarr_residence_time[temp_y, temp_x]>0:
                    if temp_tup not in burned_cells:
                        burned_cells.append(temp_tup)
            with open(PICKLE_PATH, 'wb') as f:
                pickle.dump(burned_cells,f)
            print('While loop complete.')
        else: #reload previous list
            with open(PICKLE_PATH, 'rb') as f:
                burned_cells = pickle.load(f)
        return burned_cells 
    
    #Graph power overtime
    def build_power_graph(power, x_cell, y_cell, roll_avg, save_dir=save_dir):
        x_cell_m = x_cell * 2
        y_cell_m = y_cell * 2
        plt.plot(range(len(power)), power)
        plt.xlabel('Time (s)')
        plt.ylabel('Power (kW/m^2)')
        plt.title('Power From Surface Cell\nx={}m, y={}m, Rolling Avg={}'.format(x_cell_m,y_cell_m,roll_avg))
        plt.savefig(os.path.join(save_dir, 'SufaceCellx-{}_y-{}_RollingAvg={}.png'.format(x_cell_m,y_cell_m,roll_avg)))
        plt.close()

    CF_PATH = os.path.join(save_dir, "Cell_Figures")
    if not os.path.exists(CF_PATH):
        os.makedirs(CF_PATH)
    # burned_cells = find_cells_that_burned(xarr_residence_time, SIM_PARAMS, n=100)
    # power_metrics = {'max_power':[],'total_eng':[]}
    # import time
    # strt_time = time.time()
    # for i, bc in enumerate(burned_cells):
    #     print(i)
    #     print(time.time()-strt_time)
    #     x_cell, y_cell = bc
    #     start_t = int(xarr_arrival_time[y_cell, x_cell])
    #     stop_t = int(xarr_fire_stop_time[y_cell, x_cell])
    #     cell_power = ds.surfEnergy[start_t:stop_t,y_cell,x_cell]
    #     cell_power = cell_power.to_numpy()
    #     df = pd.DataFrame({"cell_power_1":cell_power})
    #     roll_vals = [1, 5, 10, 15, 20]
    #     roll_names = ["cell_power_1", "cell_power_5", "cell_power_10", "cell_power_15", "cell_power_20"]
    #     for i in range(len(roll_vals)):
    #         rn = roll_names[i]
    #         rv = roll_vals[i]
    #         if i == 0:
    #             cp = cell_power
    #         else:
    #             df[rn]=df[roll_names[0]].rolling(rv, min_periods=1).mean()
    #             cp = np.array(df[rn])
    #         temp_sav_dir = os.path.join(CF_PATH, rn)
    #         if not os.path.exists(temp_sav_dir):
    #             os.makedirs(temp_sav_dir)
    #         build_power_graph(cp, x_cell, y_cell, rv, temp_sav_dir)
    #     power_metrics["max_power"].append(float(xarr_max_power[y_cell, x_cell]))
    #     power_metrics["total_eng"].append(float(cell_power.sum()))
    #     np.savetxt(os.path.join(temp_sav_dir,'SufaceCellx-{}_y-{}.csv'.format(x_cell*2,y_cell*2)), cell_power, delimiter=",")
    # df = pd.DataFrame(power_metrics)
    # df.to_csv(os.path.join(save_dir,'power_metrics.csv'))

    # plt.hist(df['max_power'])
    # plt.title('Maximum Power for Selected Cells')
    # plt.xlabel('Max Power kW/m^2')
    # plt.ylabel('Frequency')
    # plt.savefig(os.path.join(CF_PATH,'max_power_hist_select_cells.png'))
    # plt.close()

    # plt.hist(df['total_eng'])
    # plt.title('Total Energy for Selected Cells')
    # plt.xlabel('Total Energy kJ/m^2')
    # plt.ylabel('Frequency')
    # plt.savefig(os.path.join(CF_PATH,'total_eng_hist_select_cells.png'))
    # plt.close()

    


In [22]:
#Setup drawfire folder:
if not os.path.exists(DRAWFIRE_PATH):
    os.makedirs(DRAWFIRE_PATH)
save_dir = DRAWFIRE_PATH
SMOLDER_THRESHOLD = 25
#Use library to load and calculate surfEnergy outputs
simulation_outputs = outputs.SimulationOutputs(OUTPUT_PATH, SIM_PARAMS)

# output = simulation_outputs.get_output("surfEnergy")
# arr = simulation_outputs.to_numpy(output)

# # Create a DataArray object from the numpy array
# da = xr.DataArray(arr, dims=["time", "y", "x"])

# # Create a Dataset object from the DataArray object
# ds = da.to_dataset(name="data")
if not os.path.exists(ZARR_PATH):
    zarr_file = simulation_outputs.to_zarr(ZARR_PATH)
zarr.convenience.consolidate_metadata(ZARR_PATH)
ds = xr.open_zarr(ZARR_PATH)

#Calc time for max power
ds = ds.fillna(0) #Convert nan to 0 for dask
xarr_max_power_time = ds.surfEnergy.argmax('time')
xarr_max_power = ds.surfEnergy[xarr_max_power_time.compute()]   
xarr_max_power = xarr_max_power.compute()

#Calc Total Energy
xarr_total_energy = ds.surfEnergy.sum(dim='time')
xarr_total_energy = xarr_total_energy.compute()

###Calc Times: arrival, stop, residence
##Removed forloop to improve speed
#https://stackoverflow.com/questions/47269390/how-to-find-first-non-zero-value-in-every-column-of-a-numpy-array
#https://stackoverflow.com/questions/66305130/index-of-last-occurence-of-true-in-every-row
burned_binary = (ds>SMOLDER_THRESHOLD)
#Arrival time
xarr_arrival_time = burned_binary.surfEnergy.argmax('time')
xarr_arrival_time = xr.where(xarr_arrival_time==0,np.nan,xarr_arrival_time) #0 to nan
xarr_arrival_time = xarr_arrival_time.compute()

#Fire stop time
xarr_fire_stop_time = burned_binary.dims['time'] - burned_binary.surfEnergy[::-1,:,:].argmax('time') - 1
xarr_fire_stop_time = xr.where((burned_binary.surfEnergy[-1,:,:]==0) & (xarr_fire_stop_time==xarr_fire_stop_time.max()),np.nan,xarr_fire_stop_time) #non-burning cells to nan
xarr_fire_stop_time = xarr_fire_stop_time.compute()
del burned_binary

xarr_residence_time = xarr_fire_stop_time - xarr_arrival_time

In [4]:
###This method is dumb. I should be able to use np.where to index the burned cells:
#Sample burning cells
def find_cells_that_burned(xarr_residence_time, SIM_PARAMS, n=1, time_len=15):
    """
    xarr_residence_time: residence times
    SIM_PARAMS: class of simulation parameters
    n: # of cells to sample
    time_len: length of time to consider cell burned for sample
    """
    PICKLE_PATH = os.path.join(save_dir, 'cell_that_burned.pkl')
    if not os.path.exists(PICKLE_PATH):
        nx = SIM_PARAMS.nx
        ny = SIM_PARAMS.ny
        burned_cells = []
        print('Starting while loop')
        while len(burned_cells) < n:
            temp_x = int(nx*random.random())
            temp_y = int(ny*random.random())
            temp_tup = (temp_x, temp_y)
            if xarr_residence_time[temp_y, temp_x]>0:
                if temp_tup not in burned_cells:
                    burned_cells.append(temp_tup)
        with open(PICKLE_PATH, 'wb') as f:
            pickle.dump(burned_cells,f)
        print('While loop complete.')
    else: #reload previous list
        with open(PICKLE_PATH, 'rb') as f:
            burned_cells = pickle.load(f)
    return burned_cells 

#Graph power overtime
def build_power_graph(power, x_cell, y_cell, roll_avg, save_dir=save_dir):
    x_cell_m = x_cell * 2
    y_cell_m = y_cell * 2
    plt.plot(range(len(power)), power)
    plt.xlabel('Time (s)')
    plt.ylabel('Power (kW/m^2)')
    plt.title('Power From Surface Cell\nx={}m, y={}m, Rolling Avg={}'.format(x_cell_m,y_cell_m,roll_avg))
    plt.savefig(os.path.join(save_dir, 'SufaceCellx-{}_y-{}_RollingAvg={}.png'.format(x_cell_m,y_cell_m,roll_avg)))
    plt.close()

CF_PATH = os.path.join(save_dir, "Cell_Figures")
if not os.path.exists(CF_PATH):
    os.makedirs(CF_PATH)
# burned_cells = find_cells_that_burned(xarr_residence_time, SIM_PARAMS, n=100)
# power_metrics = {'max_power':[],'total_eng':[]}
# import time
# strt_time = time.time()
# for i, bc in enumerate(burned_cells):
#     print(i)
#     print(time.time()-strt_time)
#     x_cell, y_cell = bc
#     start_t = int(xarr_arrival_time[y_cell, x_cell])
#     stop_t = int(xarr_fire_stop_time[y_cell, x_cell])
#     cell_power = ds.surfEnergy[start_t:stop_t,y_cell,x_cell]
#     cell_power = cell_power.to_numpy()
#     df = pd.DataFrame({"cell_power_1":cell_power})
#     roll_vals = [1, 5, 10, 15, 20]
#     roll_names = ["cell_power_1", "cell_power_5", "cell_power_10", "cell_power_15", "cell_power_20"]
#     for i in range(len(roll_vals)):
#         rn = roll_names[i]
#         rv = roll_vals[i]
#         if i == 0:
#             cp = cell_power
#         else:
#             df[rn]=df[roll_names[0]].rolling(rv, min_periods=1).mean()
#             cp = np.array(df[rn])
#         temp_sav_dir = os.path.join(CF_PATH, rn)
#         if not os.path.exists(temp_sav_dir):
#             os.makedirs(temp_sav_dir)
#         build_power_graph(cp, x_cell, y_cell, rv, temp_sav_dir)
#     power_metrics["max_power"].append(float(xarr_max_power[y_cell, x_cell]))
#     power_metrics["total_eng"].append(float(cell_power.sum()))
#     np.savetxt(os.path.join(temp_sav_dir,'SufaceCellx-{}_y-{}.csv'.format(x_cell*2,y_cell*2)), cell_power, delimiter=",")
# df = pd.DataFrame(power_metrics)
# df.to_csv(os.path.join(save_dir,'power_metrics.csv'))

# plt.hist(df['max_power'])
# plt.title('Maximum Power for Selected Cells')
# plt.xlabel('Max Power kW/m^2')
# plt.ylabel('Frequency')
# plt.savefig(os.path.join(CF_PATH,'max_power_hist_select_cells.png'))
# plt.close()

# plt.hist(df['total_eng'])
# plt.title('Total Energy for Selected Cells')
# plt.xlabel('Total Energy kJ/m^2')
# plt.ylabel('Frequency')
# plt.savefig(os.path.join(CF_PATH,'total_eng_hist_select_cells.png'))
# plt.close()

In [31]:
#Build Figures
def scale_for_figs_x_and_y(arr, dx=2, dy=2):
    arr = np.array(arr)
    arr = np.repeat(np.repeat(arr, dy, axis=0), dx, axis=1)
    plt.imshow(arr, cmap='YlOrRd', origin="lower")

def build_hist_remove_zeros(arr, bins=25, MAX=None):
    arr = np.array(arr)
    arr = arr[arr>0]
    if MAX is None:
        MAX = arr.max()
    plt.hist(arr, bins=bins, range=(arr.min(),MAX))
    return arr

#Plot Spatial metrics
scale_for_figs_x_and_y(xarr_arrival_time)
plt.colorbar()
plt.title("Arrival Time (s)")
plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.savefig(os.path.join(save_dir,"arrival_time.png"))
plt.close()

#Spatial Figures of Metrics
scale_for_figs_x_and_y(xarr_fire_stop_time)
plt.colorbar()
plt.title("Burn Completion Time (s)")
plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.savefig(os.path.join(save_dir,"stop_time.png"))
plt.close()

In [32]:
scale_for_figs_x_and_y(xarr_residence_time)
plt.colorbar()
plt.title("Residence Time (s)")
plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.savefig(os.path.join(save_dir,"residence_time.png"))
plt.close()
residence_time_cleaned = build_hist_remove_zeros(xarr_residence_time, MAX=400)
plt.title('Residence Time Histogram')
plt.xlabel('Residence Time (s)')
plt.ylabel('Frequency')
plt.savefig(os.path.join(save_dir,'residence_time_hist.png'))
plt.close()
np.savetxt(os.path.join(save_dir,'residence_time.csv'), residence_time_cleaned, delimiter=",")

In [ ]:
xarr_total_energy = xarr_total_energy.compute()

In [33]:
scale_for_figs_x_and_y(xarr_total_energy)
plt.colorbar()
plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.title("Total Energy (kJ/m^2)")
plt.savefig(os.path.join(save_dir,"total_energy.png"))
plt.close()
total_energy_cleaned = build_hist_remove_zeros(xarr_total_energy)
plt.title('Total Energy Histogram')
plt.xlabel('Total Energy (kJ/m^2)')
plt.ylabel('Frequency')
plt.savefig(os.path.join(save_dir,'total_energy_hist.png'))
plt.close()
np.savetxt(os.path.join(save_dir,'total_energy.csv'), total_energy_cleaned, delimiter=",")

In [35]:
xarr_max_power = xarr_max_power.compute()

In [36]:
scale_for_figs_x_and_y(xarr_max_power)
plt.colorbar()
plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.title("Max Power (kW/m^2)")
plt.savefig(os.path.join(save_dir,"max_power.png"))
plt.close()
max_power_cleaned = build_hist_remove_zeros(xarr_max_power, MAX=1500)
plt.title('Max Power Histogram')
plt.xlabel('Max Power (kW/m^2)')
plt.ylabel('Frequency')
plt.savefig(os.path.join(save_dir,'max_power_hist.png'))
plt.close()
np.savetxt(os.path.join(save_dir,'max_power.csv'), max_power_cleaned, delimiter=",")

In [82]:
NUM_CELLS_SAMPLE = 10000

#NEW code to run for avg power and stdDev:
burn_indexes = np.where(xarr_residence_time>0)
num_burn_cells = len(burn_indexes[0])
#max_res_time = xarr_residence_time.max()
import random
sample_cell_indexes = random.sample(range(num_burn_cells), NUM_CELLS_SAMPLE)
max_res_time = 400
power_burn_cells = np.empty((NUM_CELLS_SAMPLE,max_res_time))
power_burn_cells[:] = np.nan

In [92]:
for i,sample_index in enumerate(sample_cell_indexes):#range(num_burn_cells):
    ty = burn_indexes[0][sample_index] #temp y
    tx = burn_indexes[1][sample_index]
    start_t = int(xarr_arrival_time[ty, tx])
    stop_t = int(xarr_fire_stop_time[ty, tx])
    temp_res_time = stop_t-start_t
    if temp_res_time>max_res_time:
        diff = temp_res_time-max_res_time
        stop_t = stop_t-diff
    cell_power = ds.surfEnergy[start_t:stop_t,ty,tx]
    power_burn_cells[i,:(stop_t-start_t)] = cell_power.to_numpy()

In [94]:
power_mean_overtime = np.nanmean(power_burn_cells, axis=0)
power_median_overtime = np.nanmedian(power_burn_cells, axis=0)
power_stdev_overtime = np.nanstd(power_burn_cells, axis=0)

#Raw data for Joe:
df = pd.DataFrame({"power_mean_overtime":power_mean_overtime,"power_median_overtime":power_median_overtime,
                    "power_stdev_overtime":power_stdev_overtime})
df.to_csv(os.path.join(save_dir,"FtStewart_PowerOvertime.csv"), index=False)
del df

plt.plot(range(max_res_time), power_mean_overtime)
plt.fill_between(range(max_res_time),power_mean_overtime-power_stdev_overtime,power_mean_overtime+power_stdev_overtime, alpha=0.2)
plt.xlabel('Time (s)')
plt.ylabel('Power (kW/m^2)')
plt.title('Average Power of Surface Cells After Ignition')
plt.ylim(bottom=0)
plt.savefig(os.path.join(save_dir, 'SufaceCell_MeanPowerAfterIg.png'))
plt.close()  

plt.plot(range(max_res_time), power_median_overtime)
plt.fill_between(range(max_res_time),power_median_overtime-power_stdev_overtime,power_median_overtime+power_stdev_overtime, alpha=0.2)
plt.xlabel('Time (s)')
plt.ylabel('Power (kW/m^2)')
plt.title('Median Power of Surface Cells After Ignition')
plt.ylim(bottom=0)
plt.savefig(os.path.join(save_dir, 'SufaceCell_MedianPowerAfterIg.png'))
plt.close()  